### RAG Pipline -Data ingestion to vector DB Pipline

In [41]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### read all pdf files in a directory
def process_all_pdfs(pdf_direcort):
    all_documents = []
    pdf_direcort=Path(pdf_direcort)
    pdf_files = list(pdf_direcort.glob("**/*.pdf"))
    print(f"Processing all pdf files in {len(pdf_files)}")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["page"] = doc.metadata.get("page", None)
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name} using PyPDFLoader.")
        except Exception as e:
            print(f"Failed to load {pdf_file.name} with PyPDFLoader: {e}. Trying PyMuPDFLoader.")

        loader = PyMuPDFLoader(str(pdf_file))
        documents = loader.load()
        all_documents.extend(documents)
    return all_documents
all_documents = process_all_pdfs("../data/pdf")





In [ ]:
all_documents

In [44]:
### Text Splitter
def create_text_splitter(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    splitter = text_splitter.split_documents(documents)

    print(f"Total split documents: {len(splitter)} chunks")

    if splitter:
        print(f"First chunk: {splitter[0].page_content[:100]}...")
        print(f"Last chunk: {splitter[-1].page_content[:100]}...")
        print(f"First chunk metadata: {splitter[0].metadata}")
        print(f"Last chunk metadata: {splitter[-1].metadata}")

    return splitter


In [ ]:
chunk = create_text_splitter(all_documents)
chunk

In [46]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
class EmbeddingModel:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded model: {self.model_name}")
        except Exception as e:
            print(f"Failed to load model {self.model_name}: {e}")
            raise

    def embed(self, texts: List[str]) -> np.ndarray:
        print(f"Embedding {len(texts)} texts using model: {self.model_name}")

        embeddings = self.model.encode(
            texts,
            convert_to_numpy=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        return self.model.get_sentence_embedding_dimension()
embeddings_model = EmbeddingModel()
embeddings_model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1517.78it/s]


Loaded model: all-MiniLM-L6-v2


### Vector Store

In [48]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [51]:
### convert the text to embeddings and add to vector store
texts = [doc.page_content for doc in chunk]

embeddings = embeddings_model.embed(texts)
vectorstore.add_documents(chunk, embeddings) 

Embedding 192 texts using model: all-MiniLM-L6-v2
Generated embeddings with shape: (192, 384)
Adding 192 documents to vector store...
Successfully added 192 documents to vector store
Total documents in collection: 192
